In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [31]:
load_dotenv(override=True)

True

In [147]:
# Constants 
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI, api_key
import os

api_key = os.getenv("OPENROUTER_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
openrouter_client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key)
gemini_client = AsyncOpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=gemini_api_key)
openrouter_model = OpenAIChatCompletionsModel(openai_client=openrouter_client,model="openrouter/free")
gemini_model = OpenAIChatCompletionsModel(openai_client=gemini_client,model="gemini-3.5-flash-lite")
gemma_model = OpenAIChatCompletionsModel(openai_client=openrouter_client,model="deepseek/deepseek-chat-v3.1:free")
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

tavily_api_key = os.getenv("TAVILY_API_KEY")

print(tavily_api_key is not None)

True


In [15]:
from agents import function_tool
from tavily import TavilyClient
import os

tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)

@function_tool
def web_search(query:str)->str:

    response = tavily_client.search(
      query=query,
      max_results=5,
      search_depth="advanced"
    )


    results = []

    for result in response["results"]:
        results.append(
            f"Title: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"Content: {result['content']}\n"
        )

    return "\n\n".join(results)

In [137]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Tell me about most popular AI agent frameworks in 2026"
settings = ModelSettings(tool_choice="required")

In [ ]:
search_agent = Agent(name = "search_agent",instructions = INSTRUCTIONS,model = gemini_model,tools = [web_search],model_settings=settings)



In [ ]:
result = await Runner.run(search_agent,task)


In [ ]:
display(Markdown(result.final_output))

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields 

In [22]:
class WebSearchItem(BaseModel):
  reason:str = Field(description="Your reasoning for why this search is important to the query.")
  query:str = Field(description="The search term to use for the web search.")
class WebSearchPlan(BaseModel):
  searches:list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [23]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [149]:
P_INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""
planner_agent = Agent(name="planner Agent",instructions=P_INSTRUCTIONS,model = gemini_model,output_type=WebSearchPlan)

In [37]:
result = await Runner.run(planner_agent,task)

In [40]:
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Identify the leading and most widely used AI agent frameworks projected or trending into 2026.', query='most popular AI agent frameworks 2025 2026'), WebSearchItem(reason='Examine top open-source frameworks like LangGraph, CrewAI, AutoGen, and newer contenders.', query='best open source AI agent frameworks comparison LangGraph CrewAI AutoGen'), WebSearchItem(reason='Discover newly emerging agentic frameworks and multi-agent systems gaining traction.', query='emerging multi agent AI frameworks architecture 2025 2026'), WebSearchItem(reason='Find industry reports, GitHub stars, and developer adoption trends for AI agent development libraries.', query='AI agent frameworks developer adoption trends 2025 2026'), WebSearchItem(reason='Look for enterprise-focused AI agent platforms and orchestrators popular in modern production deployments.', query='enterprise AI agent orchestration platforms tools 2026')])

## Agent 3: The Writer Agent

In [151]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=gemini_model, output_type=ReportData)

## Agent 4: The email agent

In [42]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [43]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [57]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=openrouter_model)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [55]:
async def Run_Searches(query: str):
    print("Planning Searches...")

    result = await Runner.run(
        planner_agent,
        f"Query: {query}"
    )

    searches = result.final_output.searches

    print(f"Will perform {len(searches)} searches")

    tasks = [search(item) for item in searches]

    results = await asyncio.gather(*tasks)

    print("Finished searching")

    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [47]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

# showtime

In [152]:
query = "Most popular agent frameworks in 2026"

with trace("Reasearch Agent"):
  print("Starting Research")
  search_results = await Run_Searches(query)
  report = await write_report(query,search_results)
  await send_report_email(report)
  print("Hooray the research is completed")

Starting Research
Planning Searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray the research is completed
